# RSNA — varianza entre semillas y auditoría Grad-CAM con potencia

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental.

Cierra dos debilidades del primer experimento:

1. **Una sola semilla no permite interpretar diferencias.** Se entrenan tres
   modelos idénticos salvo la inicialización y el orden de los datos. La
   desviación entre ellos es el suelo de ruido: cualquier diferencia menor que
   eso no significa nada.
2. **La auditoría de atajos con 16 imágenes no sostiene ninguna afirmación.**
   Aquí se hace sobre 400, separando detecciones positivas del resto.

Código: https://github.com/GGGuardin/chest-xray-pneumonia

In [ ]:
import subprocess, sys, os, time, glob, json
T0 = time.time()

subprocess.run(['rm', '-rf', '/tmp/repo'], check=False)
subprocess.run(['git', 'clone', '--depth', '1', '-q',
                'https://github.com/GGGuardin/chest-xray-pneumonia.git', '/tmp/repo'], check=True)
os.chdir('/tmp/repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations', 'pydicom'], check=True)

import torch
cap = torch.cuda.get_device_capability(0)
assert f'sm_{cap[0]}{cap[1]}' in torch.cuda.get_arch_list(), 'GPU no soportada, relanza con T4'
torch.zeros(8, device='cuda').sum().item()
print('GPU:', torch.cuda.get_device_name(0), '| CUDA OK')

In [ ]:
OUT = '/kaggle/working'
anclas = glob.glob('/kaggle/input/**/stage_2_train_labels.csv', recursive=True)
RSNA = os.path.dirname(anclas[0])
nih_csv = glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
NIH = os.path.dirname(nih_csv[0]) if nih_csv else None
print('RSNA:', RSNA, '| NIH:', NIH)

# El split es determinista (semilla fija en make_splits), asi que los tres
# modelos comparten exactamente el mismo train/val/test: lo unico que varia
# entre ellos es la inicializacion y el orden de los lotes.
!python -m src.prepare_data --dataset rsna --root {RSNA} --out {OUT}/manifest_rsna.csv

## 1. Tres semillas

In [ ]:
SEMILLAS = [42, 1337, 2024]

for s in SEMILLAS:
    print(f'\n===== semilla {s} =====', flush=True)
    !python -m src.train --config configs/rsna.yaml \
        --manifest {OUT}/manifest_rsna.csv \
        --out-dir {OUT}/runs/rsna_s{s} --batch-size 64 --seed {s}
    !python -m src.evaluate --checkpoint {OUT}/runs/rsna_s{s}/best.pth \
        --manifest {OUT}/manifest_rsna.csv --split test \
        --out-dir {OUT}/reports/s{s}_test --n-boot 500

In [ ]:
import numpy as np

aurocs, auprcs = [], []
for s in SEMILLAS:
    with open(f'{OUT}/reports/s{s}_test/metrics.json') as f:
        m = json.load(f)
    aurocs.append(m['auroc']); auprcs.append(m['auprc'])
    print(f"semilla {s}: AUROC {m['auroc']:.4f}  AUPRC {m['auprc']:.4f}")

varianza = {
    'semillas': SEMILLAS,
    'auroc': {'valores': aurocs, 'media': float(np.mean(aurocs)),
              'desviacion': float(np.std(aurocs, ddof=1)),
              'rango': float(max(aurocs) - min(aurocs))},
    'auprc': {'valores': auprcs, 'media': float(np.mean(auprcs)),
              'desviacion': float(np.std(auprcs, ddof=1)),
              'rango': float(max(auprcs) - min(auprcs))},
}
with open(f'{OUT}/varianza_semillas.json', 'w') as f:
    json.dump(varianza, f, indent=2)
print('\nAUROC: %.4f +- %.4f (rango %.4f)' % (
    varianza['auroc']['media'], varianza['auroc']['desviacion'], varianza['auroc']['rango']))
print('Cualquier diferencia por debajo de ese rango no es interpretable.')

## 2. Comparación pareada sobre el test de NIH

El experimento anterior comparaba el modelo de RSNA sobre NIH completo contra
el modelo de NIH sobre su test retenido: dos conjuntos distintos. Aquí se
genera el manifiesto de NIH con **la misma semilla de split**, y se evalúa el
modelo de RSNA **sobre ese mismo test**, de modo que la comparación sea pareada
imagen a imagen.

In [ ]:
if NIH:
    !python -m src.prepare_data --dataset nih --root {NIH} --target Pneumonia \
        --out {OUT}/manifest_nih.csv
    for s in SEMILLAS:
        !python -m src.evaluate --checkpoint {OUT}/runs/rsna_s{s}/best.pth \
            --manifest {OUT}/manifest_nih.csv --split test \
            --out-dir {OUT}/reports/s{s}_nih_test --n-boot 500
else:
    print('NIH no montado')

## 3. Auditoría de atajos con potencia

400 imágenes en vez de 16. Con 4 detecciones no se puede afirmar nada sobre
dónde mira el modelo; con unos cientos, sí.

In [ ]:
!python -m src.explain --checkpoint {OUT}/runs/rsna_s42/best.pth \
    --manifest {OUT}/manifest_rsna.csv --split test --n 400 \
    --out-dir {OUT}/reports/gradcam_400

# Los 400 PNG no aportan como salida; se conserva solo el JSON del audit
import shutil
for p in glob.glob(f'{OUT}/reports/gradcam_400/*.png'):
    os.remove(p)
print('mapas borrados, audit conservado')

In [ ]:
import shutil
for f_ in glob.glob(f'{OUT}/manifest_*.csv'):
    shutil.move(f_, '/tmp/' + os.path.basename(f_))
for f_ in glob.glob(f'{OUT}/runs/*/best.pth'):
    if 'rsna_s42' not in f_:  # solo se conserva un checkpoint
        os.remove(f_)
print('minutos totales: %.1f' % ((time.time() - T0) / 60))
for p in sorted(glob.glob(f'{OUT}/**/*', recursive=True)):
    if os.path.isfile(p):
        print(f'  {os.path.getsize(p)/1e6:8.2f} MB  {p}')